## Import all libraries needed to begin project.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import random
from pathlib import Path
from datetime import date, timedelta

np.random.seed(42)
random.seed(42)

OUTPUT = Path('output')
OUTPUT.mkdir(exist_ok=True)
DB_PATH = OUTPUT / 'meadowtable.db'

N_LOCATIONS  = 20
START_DATE   = date(2023, 1, 1)
END_DATE     = date(2024, 6, 30)

MARKETS = {
    'Orlando':    5,
    'Tampa':      4,
    'Jacksonville': 4,
    'Miami':      4,
    'Tallahassee': 3,
}

DAYPARTS = ['Lunch', 'Dinner']

AVG_CHECK        = 18.50
LABOR_COST_TARGET = 0.30
HOURLY_WAGE      = 14.50
OVERTIME_RATE    = HOURLY_WAGE * 1.5

print('Config loaded.')
print(f'  Locations : {N_LOCATIONS}')
print(f'  Date range: {START_DATE} to {END_DATE}')
print(f'  Total days: {(END_DATE - START_DATE).days + 1}')

Config loaded.
  Locations : 20
  Date range: 2023-01-01 to 2024-06-30
  Total days: 547


## Create a table condensing all of the restaurants locations.

In [2]:
locations = []
loc_id = 1

for market, count in MARKETS.items():
    for i in range(count):
        perf = np.clip(np.random.normal(1.0, 0.15), 0.65, 1.35)
        locations.append({
            'location_id':   f'LOC-{loc_id:03d}',
            'store_name':    f'MeadowTable {market} #{i+1}',
            'market':        market,
            'is_franchise':  random.choice([True, False, False]),
            'seating_capacity': random.choice([80, 100, 120, 140]),
            'perf_baseline': round(perf, 3),
            'manager_tenure_yrs': round(np.random.uniform(0.5, 8.0), 1),
        })
        loc_id += 1

locations_df = pd.DataFrame(locations)
print(f'Built {len(locations_df)} locations')
print(locations_df.head(6).to_string(index=False))

Built 20 locations
location_id             store_name  market  is_franchise  seating_capacity  perf_baseline  manager_tenure_yrs
    LOC-001 MeadowTable Orlando #1 Orlando         False                80          1.075                 6.0
    LOC-002 MeadowTable Orlando #2 Orlando          True               120          0.979                 5.0
    LOC-003 MeadowTable Orlando #3 Orlando          True               100          0.965                 0.9
    LOC-004 MeadowTable Orlando #4 Orlando          True                80          0.965                 7.0
    LOC-005 MeadowTable Orlando #5 Orlando         False                80          1.237                 0.7
    LOC-006   MeadowTable Tampa #1   Tampa         False               140          1.115                 7.8


## Generate daily sales data for each location. Each row of data shows one location each day and revenue covers guest count multipled by average check. Data also includes stronger weekend sales, seasonal trends, each locations performance baseline, and random daily noise.

In [3]:
def day_of_week_modifier(dow, daypart):
    """Traffic modifier by day of week. dow: 0=Monday, 6=Sunday."""
    lunch_mods  = {0:1.0, 1:1.0, 2:1.0, 3:1.05, 4:1.1, 5:0.9, 6:0.85}
    dinner_mods = {0:0.85, 1:0.85, 2:0.90, 3:0.95, 4:1.15, 5:1.35, 6:1.25}
    return lunch_mods[dow] if daypart == 'Lunch' else dinner_mods[dow]

def seasonal_modifier(month):
    """Florida seasonal pattern — summer strong, January slow."""
    mods = {1:0.88, 2:0.92, 3:1.02, 4:1.05, 5:1.08,
            6:1.12, 7:1.10, 8:1.05, 9:0.95, 10:0.98, 11:1.05, 12:1.08}
    return mods.get(month, 1.0)

BASE_COVERS = {'Lunch': 55, 'Dinner': 90}

sales_records = []
current = START_DATE

while current <= END_DATE:
    dow      = current.weekday()
    season   = seasonal_modifier(current.month)

    for _, loc in locations_df.iterrows():
        for daypart in DAYPARTS:
            dow_mod  = day_of_week_modifier(dow, daypart)
            base     = BASE_COVERS[daypart]

            covers_raw = base * season * dow_mod * loc['perf_baseline']
            covers     = max(5, int(np.random.normal(covers_raw, covers_raw * 0.10)))

            avg_check  = np.random.normal(AVG_CHECK, 1.50)
            revenue    = round(covers * avg_check, 2)

            sales_records.append({
                'date':        current.isoformat(),
                'location_id': loc['location_id'],
                'daypart':     daypart,
                'day_of_week': current.strftime('%A'),
                'month':       current.month,
                'year':        current.year,
                'year_month':  current.strftime('%Y-%m'),
                'covers':      covers,
                'revenue':     revenue,
                'avg_check':   round(avg_check, 2),
            })
    current += timedelta(days=1)

sales_df = pd.DataFrame(sales_records)
print(f'Sales records: {len(sales_df):,}')
print(f'Revenue range: ${sales_df["revenue"].min():.0f} – ${sales_df["revenue"].max():.0f}')
sales_df.head(4)

Sales records: 21,880
Revenue range: $410 – $4473


,date,location_id,daypart,day_of_week,month,year,year_month,covers,revenue,avg_check
0,2023-01-01,LOC-001,Lunch,Sunday,1,2023,2023-01,47,783.43,16.67
1,2023-01-01,LOC-001,Dinner,Sunday,1,2023,2023-01,108,1680.53,15.56
2,2023-01-01,LOC-002,Lunch,Sunday,1,2023,2023-01,34,639.04,18.80
3,2023-01-01,LOC-002,Dinner,Sunday,1,2023,2023-01,104,1950.73,18.76


## Generated simulated labor data based on scheduled covers for each restaurant. The model focuses on target hours based on revenue, hours that were actually scheduled, actual scheduled hours worked, and labor costs. Data is meant to reflect that managers frequently overschedule or underschedule.

In [4]:
loc_scheduling_bias = {
    loc_id: np.random.normal(0, 0.08)
    for loc_id in locations_df['location_id']
}

labor_records = []

for _, sale in sales_df.iterrows():
    target_labor_cost = sale['revenue'] * LABOR_COST_TARGET
    target_hours      = round(target_labor_cost / HOURLY_WAGE, 1)

    bias       = loc_scheduling_bias[sale['location_id']]
    sched_raw  = target_hours * (1 + bias + np.random.normal(0, 0.05))
    sched_hrs  = round(max(2.0, sched_raw), 1)

    callout    = np.random.random() < 0.08
    actual_hrs = round(sched_hrs * np.random.normal(0.97 if callout else 1.0, 0.03), 1)
    actual_hrs = max(2.0, actual_hrs)

    overtime_hrs  = max(0, actual_hrs - 8 * 3)
    regular_hrs   = actual_hrs - overtime_hrs
    labor_cost    = round(regular_hrs * HOURLY_WAGE + overtime_hrs * OVERTIME_RATE, 2)

    labor_pct     = round(labor_cost / sale['revenue'] * 100, 1) if sale['revenue'] > 0 else 0
    variance_hrs  = round(actual_hrs - target_hours, 1)
    variance_cost = round(variance_hrs * HOURLY_WAGE, 2)

    labor_records.append({
        'date':          sale['date'],
        'location_id':   sale['location_id'],
        'daypart':       sale['daypart'],
        'target_hours':  target_hours,
        'scheduled_hrs': sched_hrs,
        'actual_hrs':    actual_hrs,
        'overtime_hrs':  round(overtime_hrs, 1),
        'labor_cost':    labor_cost,
        'labor_pct':     labor_pct,
        'variance_hrs':  variance_hrs,
        'variance_cost': variance_cost,
        'had_callout':   int(callout),
    })

labor_df = pd.DataFrame(labor_records)
print(f'Labor records: {len(labor_df):,}')
print(f'Avg labor %: {labor_df["labor_pct"].mean():.1f}% (target: {LABOR_COST_TARGET*100:.0f}%)')
print(f'Total variance cost: ${labor_df["variance_cost"].sum():,.0f}')
labor_df.head(4)

Labor records: 21,880
Avg labor %: 32.5% (target: 30%)
Total variance cost: $-5,604


,date,location_id,daypart,target_hours,scheduled_hrs,actual_hrs,overtime_hrs,labor_cost,labor_pct,variance_hrs,variance_cost,had_callout
0,2023-01-01,LOC-001,Lunch,16.2,16.1,15.7,0.0,227.65,29.1,-0.5,-7.25,0
1,2023-01-01,LOC-001,Dinner,34.8,40.3,38.6,14.6,665.55,39.6,3.8,55.10,0
2,2023-01-01,LOC-002,Lunch,13.2,12.7,13.0,0.0,188.50,29.5,-0.2,-2.90,0
3,2023-01-01,LOC-002,Dinner,40.4,35.7,37.5,13.5,641.62,32.9,-2.9,-42.05,0


## Connecting sqlite and generated three relational tables

In [5]:
conn = sqlite3.connect(DB_PATH)

locations_df.to_sql('locations', conn, if_exists='replace', index=False)
sales_df.to_sql('sales',         conn, if_exists='replace', index=False)
labor_df.to_sql('labor',         conn, if_exists='replace', index=False)

conn.close()
print(f'Database written: {DB_PATH}')
print(f'  locations : {len(locations_df):,} rows')
print(f'  sales     : {len(sales_df):,} rows')
print(f'  labor     : {len(labor_df):,} rows')

Database written: output/meadowtable.db
  locations : 20 rows
  sales     : 21,880 rows
  labor     : 21,880 rows


## Verified queries with SQL.

In [6]:
conn = sqlite3.connect(DB_PATH)

query = """
    SELECT  l.market,
            COUNT(DISTINCT s.location_id)       AS locations,
            ROUND(SUM(s.revenue), 0)            AS total_revenue,
            ROUND(AVG(s.covers), 1)             AS avg_daily_covers,
            ROUND(AVG(lb.labor_pct), 1)         AS avg_labor_pct
    FROM    sales s
    JOIN    locations l  ON s.location_id = l.location_id
    JOIN    labor lb     ON s.location_id = lb.location_id
                        AND s.date        = lb.date
                        AND s.daypart     = lb.daypart
    GROUP   BY l.market
    ORDER   BY total_revenue DESC
"""

print('Market Summary:')
print(pd.read_sql(query, conn).to_string(index=False))
conn.close()

Market Summary:
      market  locations  total_revenue  avg_daily_covers  avg_labor_pct
     Orlando          5      7934264.0              78.4           31.6
       Tampa          4      6330486.0              78.2           33.2
Jacksonville          4      5815591.0              71.9           35.7
       Miami          4      5799041.0              71.6           33.1
 Tallahassee          3      4221946.0              69.7           27.8
